# CODI/KaVa controls and additional seeds — Colab + Drive

This notebook closes the remaining Phase-2 controls before the supervision-continuum stage. Every experiment starts from the pinned GPT-2 backbone and has an isolated Drive directory. No upload is required, and the completed seed-zero CODI/KaVa checkpoints cannot be overwritten.

Run **one experiment at a time**. The training cell stays active on Colab's server, so the browser may be closed after training begins. Colab may still terminate the VM; rerunning the same experiment resumes its newest Drive checkpoint.

In [ ]:
EXPERIMENT = "latent_nodistill_seed0"
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Pin this to the pushed commit SHA for every experiment.
REPO_DIR = "/content/latent-reasoning"
DRIVE_ROOT = "/content/drive/MyDrive/CODI_KAVA"
LOCAL_ROOT = "/content/codikava_runtime"
MAX_SECONDS = 32400
EVAL_LIMIT = 200
RUN_TRAINING = True
RUN_FINAL_ANALYSIS = False

EXPERIMENT_ORDER = [
    "latent_nodistill_seed0",
    "kava_random_seed0",
    "kava_uniform_seed0",
    "codi_seed1",
    "kava_seed1",
    "codi_seed2",
    "kava_seed2",
]
assert EXPERIMENT in EXPERIMENT_ORDER


## 1. Mount Drive and pin the repository

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import json, os, pathlib, subprocess, sys
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
pathlib.Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", RUN_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")], check=True)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main":
    print("Before the first long run, pin RUN_COMMIT to:", commit)


## 2. Validate the locked experiment matrix

In [ ]:
import torch
if RUN_TRAINING:
    assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU"
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__)
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU analysis only")
reports = pathlib.Path(DRIVE_ROOT) / "reports"
reports.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "scripts/validate_controls.py", "--output", str(reports / "control_matrix_validation.json")], cwd=REPO_DIR, check=True)
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO_DIR, check=True)


## 3. Start or resume the selected experiment

Wait for `[restore] no durable checkpoint; starting fresh` or `[resume] continuing from step ...`. You may then close the browser. Exit code `42` means the wall-clock guard saved a durable checkpoint; rerun this notebook later without changing `EXPERIMENT`. Exit code `0` means training and the capped evaluation are complete.

In [ ]:
if RUN_TRAINING:
    import datetime, time
    logs = pathlib.Path(DRIVE_ROOT) / "logs" / "controls_and_seeds"
    logs.mkdir(parents=True, exist_ok=True)
    log_path = logs / f"{EXPERIMENT}.log"
    cmd = [
        sys.executable, "-u", "scripts/colab_control_runner.py",
        "--experiment", EXPERIMENT,
        "--drive-root", DRIVE_ROOT,
        "--local-root", LOCAL_ROOT,
        "--max-seconds", str(MAX_SECONDS),
        "--eval-limit", str(EVAL_LIMIT),
        "--allow-environment-change",
    ]
    print("Starting:", " ".join(cmd), flush=True)
    print("Persistent log:", log_path, flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} {' '.join(cmd)} ===\n")
        process = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        last_flush = time.monotonic()
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            if time.monotonic() - last_flush >= 30:
                log.flush()
                last_flush = time.monotonic()
        return_code = process.wait()
        log.flush()
    print("Session exit code:", return_code)
    print("0 = complete; 42 = durable checkpoint saved, rerun this experiment")
else:
    print("Training skipped.")


## 4. Inspect the selected experiment and choose the next one

In [ ]:
status_path = pathlib.Path(DRIVE_ROOT) / "status" / "controls_and_seeds" / f"{EXPERIMENT}.json"
print(json.dumps(json.loads(status_path.read_text()), indent=2) if status_path.is_file() else "No status file")
drive_output = pathlib.Path(DRIVE_ROOT) / "outputs" / "controls_and_seeds" / EXPERIMENT
for path in sorted(drive_output.rglob("*")):
    if path.is_file() and not path.name.endswith((".uploading", ".tmp")):
        print(f"{path.relative_to(drive_output)}  {path.stat().st_size / 2**20:.1f} MiB")
current = EXPERIMENT_ORDER.index(EXPERIMENT)
if current + 1 < len(EXPERIMENT_ORDER):
    print("Next experiment:", EXPERIMENT_ORDER[current + 1])
else:
    print("Training matrix complete; enable RUN_FINAL_ANALYSIS.")


## 5. Final control and multi-seed reports

Enable `RUN_FINAL_ANALYSIS` only after all seven statuses are `complete`. This section is CPU-only.

In [ ]:
if RUN_FINAL_ANALYSIS:
    from IPython.display import Markdown, display
    root = pathlib.Path(DRIVE_ROOT)
    def latest_eval(output):
        candidates = sorted((output / "eval").glob("step_*"))
        if not candidates:
            raise FileNotFoundError(f"No evaluation under {output}")
        return candidates[-1]

    primary = {method: root / "outputs" / method / "eval" / "step_00096405" / "ablations" / "baseline_bs8" for method in ("codi", "kava")}
    control_root = root / "outputs" / "controls_and_seeds"
    controls = {name: latest_eval(control_root / name) for name in EXPERIMENT_ORDER[:3]}
    control_report = reports / "control_comparison_seed0_limit200.json"
    cmd = [sys.executable, "scripts/analyze_phase2.py", "--run", f"codi={primary['codi']}", "--run", f"kava={primary['kava']}"]
    for name, path in controls.items():
        cmd.extend(["--run", f"{name}={path}"])
    cmd.extend(["--output", str(control_report)])
    subprocess.run(cmd, cwd=REPO_DIR, check=True)

    seed_report = reports / "codi_vs_kava_three_seeds_limit200.json"
    cmd = [
        sys.executable, "scripts/analyze_seed_sweep.py",
        "--run", f"codi:0={primary['codi']}",
        "--run", f"kava:0={primary['kava']}",
    ]
    for seed in (1, 2):
        cmd.extend(["--run", f"codi:{seed}={latest_eval(control_root / f'codi_seed{seed}')}"])
        cmd.extend(["--run", f"kava:{seed}={latest_eval(control_root / f'kava_seed{seed}')}"])
    cmd.extend(["--output", str(seed_report)])
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
    display(Markdown(control_report.with_suffix(".md").read_text()))
    display(Markdown(seed_report.with_suffix(".md").read_text()))
else:
    print("Final analysis skipped until all seven experiments are complete.")


## Completion rule

Do not start the supervision-granularity continuum until the three control evaluations and the paired three-seed CODI/KaVa report are saved. If compute becomes constrained, finish the four additional seed runs before adding more seeds to the controls.